In [ ]:
"""
EVA02-Base fine-tuning for ai-of-god-5 (72 animal classes, colourised segmentation overlays, macro-F1).

Key ideas
  * Colour-robust augmentation (channel permutation, strong hue/saturation jitter, grayscale, mixup/cutmix)
    so the model must rely on shape/texture, not overlay colour.
  * Same seed-42, 15% split as earlier models (auto-matches stratified/unstratified against a saved reference).
  * Layer-wise LR decay, warmup + cosine, label smoothing (via mixup targets), fp16 AMP, early stopping on val macro-F1.
  * Saves: best/last checkpoint (.pt with metadata), val+test probabilities (.npz), submission csv.

Run in a Kaggle notebook:   %run train_eva02.py        (or paste into a cell)
Requires internet ON for timm pretrained weights, OR set CFG['pretrained_file'] to an uploaded weights file.
"""
import os, json, math, random, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.optim import create_optimizer_v2

# ------------------------------------------------------------------ config
CFG = dict(
    comp_dir = '/kaggle/input/competitions/ai-of-god-5/participant_data',
    out_dir         = '/kaggle/working',
    model_name      = 'eva02_base_patch14_224.mim_in22k',
    pretrained_file = None,          # path to local weights if internet is off
    tag             = 'eva02_b',
    img_size        = 224,
    epochs          = 25,
    warmup_epochs   = 2,
    batch_size      = 32,
    lr              = 2e-4,          # top-layer LR; earlier layers get lr * layer_decay^depth
    layer_decay     = 0.75,
    weight_decay    = 0.05,
    drop_path       = 0.1,
    drop            = 0.1,
    grad_clip       = 1.0,
    label_smoothing = 0.1,
    mixup_prob      = 0.4,           # mixup/cutmix applied to 40% of batches
    patience        = 8,
    num_workers     = 4,
    seed            = 42,
    val_split       = 0.15,
    stratify        = True,          # overridden automatically if ref_state is found
    ref_state       = '/kaggle/working/ensemble_state_v1.npz',   # optional: val_ids of earlier models
    full_train      = False,         # True: train on ALL data for fixed epochs (no early stopping)
)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ------------------------------------------------------------------ data
def get_split(train, cfg):
    def split(strat):
        return train_test_split(train, test_size=cfg['val_split'], random_state=cfg['seed'],
                                stratify=train.animal_class if strat else None)
    if os.path.exists(cfg['ref_state']):
        ref = set(np.load(cfg['ref_state'], allow_pickle=True)['val_ids'].tolist())
        for strat in (True, False):
            tr, va = split(strat)
            if set(va.image_id) == ref:
                print(f'Split matches earlier models (stratify={strat})'); return tr, va
        print('WARNING: neither split matches ref val_ids; using cfg stratify setting')
    return split(cfg['stratify'])


class RandomChannelPermute:
    """Shuffle RGB channels: destroys the (arbitrary) overlay colour, keeps shape/texture."""
    def __init__(self, p=0.7): self.p = p
    def __call__(self, img):
        if random.random() < self.p:
            ch = img.split(); perm = random.sample(range(3), 3)
            img = Image.merge('RGB', [ch[i] for i in perm])
        return img


def build_transforms(size, mean, std):
    train_tf = T.Compose([
        T.RandomResizedCrop(size, scale=(0.5, 1.0), ratio=(0.75, 1.33),
                            interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        RandomChannelPermute(0.7),
        T.RandomApply([T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.5, hue=0.5)], p=0.9),
        T.RandomGrayscale(p=0.25),
        T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.15),
        T.ToTensor(),
        T.Normalize(mean, std),
        T.RandomErasing(p=0.25, scale=(0.02, 0.15), value='random'),
    ])
    eval_tf = T.Compose([
        T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(), T.Normalize(mean, std),
    ])
    return train_tf, eval_tf


class DS(Dataset):
    def __init__(self, ids, root, tf, labels=None):
        self.ids, self.root, self.tf, self.labels = list(ids), root, tf, labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        x = self.tf(Image.open(f'{self.root}/{self.ids[i]}').convert('RGB'))
        return (x, self.labels[i]) if self.labels is not None else x


# ------------------------------------------------------------------ model
def make_model(cfg, n_classes):
    m = timm.create_model(cfg['model_name'], pretrained=cfg['pretrained_file'] is None,
                          num_classes=n_classes, drop_path_rate=cfg['drop_path'], drop_rate=cfg['drop'])
    if cfg['pretrained_file']:
        sd = torch.load(cfg['pretrained_file'], map_location='cpu')
        sd = sd.get('state_dict', sd.get('model', sd))
        msd = m.state_dict()
        sd = {k: v for k, v in sd.items() if k in msd and v.shape == msd[k].shape}
        print('loaded local pretrained:', m.load_state_dict(sd, strict=False))
    return m


@torch.no_grad()
def predict(model, loader, tta=True):
    model.eval(); out = []
    for x in loader:
        if isinstance(x, (list, tuple)): x = x[0]
        x = x.to(dev, non_blocking=True)
        with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
            p = model(x).float().softmax(1)
            if tta: p = (p + model(x.flip(3)).float().softmax(1)) / 2
        out.append(p.cpu())
    return torch.cat(out).numpy()


def load_for_inference(path):
    """Use later:  model, ck = load_for_inference('/kaggle/working/eva02_b_best.pt')"""
    ck = torch.load(path, map_location='cpu', weights_only=False)
    m = timm.create_model(ck['model_name'], pretrained=False, num_classes=len(ck['classes']))
    m.load_state_dict(ck['state_dict']); return m.to(dev).eval(), ck


# ------------------------------------------------------------------ main
def main(cfg=CFG):
    set_seed(cfg['seed'])
    comp, out = cfg['comp_dir'], cfg['out_dir']
    train = pd.read_csv(f'{comp}/train.csv')
    classes = sorted(train.animal_class.unique()); assert len(classes) == 72
    cmap = {c: i for i, c in enumerate(classes)}
    tr, va = get_split(train, cfg)
    if cfg['full_train']: tr = train
    print(f'train {len(tr)}  val {len(va)}  full_train={cfg["full_train"]}')

    model = make_model(cfg, 72).to(dev)
    pc = model.pretrained_cfg; mean, std = pc['mean'], pc['std']
    print('normalisation', mean, std)
    train_tf, eval_tf = build_transforms(cfg['img_size'], mean, std)

    img_dir = f'{comp}/train_images'
    ytr = tr.animal_class.map(cmap).values; yva = va.animal_class.map(cmap).values
    kw = dict(num_workers=cfg['num_workers'], pin_memory=True, persistent_workers=cfg['num_workers'] > 0)
    dl_tr = DataLoader(DS(tr.image_id, img_dir, train_tf, ytr), batch_size=cfg['batch_size'],
                       shuffle=True, drop_last=True, **kw)
    dl_va = DataLoader(DS(va.image_id, img_dir, eval_tf, yva), batch_size=64, **kw)

    mix = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=cfg['mixup_prob'], switch_prob=0.5,
                label_smoothing=cfg['label_smoothing'], num_classes=72)
    crit = SoftTargetCrossEntropy()
    opt = create_optimizer_v2(model, opt='adamw', lr=cfg['lr'], weight_decay=cfg['weight_decay'],
                              layer_decay=cfg['layer_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=dev == 'cuda')

    steps_per_ep = len(dl_tr); total = cfg['epochs'] * steps_per_ep; warm = cfg['warmup_epochs'] * steps_per_ep
    def lr_at(step):
        if step < warm: return cfg['lr'] * (step + 1) / warm
        return 1e-6 + 0.5 * (cfg['lr'] - 1e-6) * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))

    best_f1, best_ep, bad, step = -1, -1, 0, 0
    best_path = f'{out}/{cfg["tag"]}_best.pt'; last_path = f'{out}/{cfg["tag"]}_last.pt'
    meta = dict(model_name=cfg['model_name'], classes=classes, img_size=cfg['img_size'],
                mean=mean, std=std, cfg=cfg)
    history = []

    for ep in range(cfg['epochs']):
        model.train(); t0 = time.time(); run = 0.0
        for x, y in dl_tr:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            x, ys = mix(x, y)
            for g in opt.param_groups: g['lr'] = lr_at(step) * g.get('lr_scale', 1.0)
            with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
                loss = crit(model(x), ys)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            scaler.step(opt); scaler.update()
            run += loss.item(); step += 1
        pv = predict(model, dl_va, tta=False)
        f1 = f1_score(yva, pv.argmax(1), average='macro'); acc = accuracy_score(yva, pv.argmax(1))
        history.append(dict(epoch=ep + 1, loss=run / steps_per_ep, val_f1=f1, val_acc=acc))
        print(f'ep {ep+1:02d}/{cfg["epochs"]}  loss {run/steps_per_ep:.4f}  val F1 {f1:.4f}  acc {acc:.4f}  '
              f'{time.time()-t0:.0f}s')
        torch.save(dict(state_dict=model.state_dict(), epoch=ep + 1, val_f1=f1, **meta), last_path)
        if f1 > best_f1 or cfg['full_train']:
            best_f1, best_ep, bad = f1, ep + 1, 0
            torch.save(dict(state_dict=model.state_dict(), epoch=ep + 1, val_f1=f1, **meta), best_path)
        else:
            bad += 1
            if bad >= cfg['patience'] and not cfg['full_train']:
                print(f'early stop (best ep {best_ep}, F1 {best_f1:.4f})'); break
    json.dump(history, open(f'{out}/{cfg["tag"]}_history.json', 'w'))
    print(f'best epoch {best_ep}  val macro-F1 {best_f1:.4f}')

    # ---- reload best, export probabilities (flip TTA) for ensembling
    model, ck = load_for_inference(best_path)
    pv = predict(model, dl_va, tta=True)
    print('best-ckpt val macro-F1 (flip TTA):', round(f1_score(yva, pv.argmax(1), average='macro'), 4))

    sample = pd.read_csv(f'{comp}/sample_submission.csv'); test_ids = sample.image_id.tolist()
    dl_te = DataLoader(DS(test_ids, f'{comp}/test_images', eval_tf), batch_size=64, **kw)
    pt = predict(model, dl_te, tta=True)

    np.savez(f'{out}/{cfg["tag"]}_probs.npz', val_ids=np.array(va.image_id.tolist()), y_val=yva,
             test_ids=np.array(test_ids), classes=np.array(classes), val_probs=pv, test_probs=pt)
    pd.DataFrame({'image_id': test_ids, 'animal_class': [classes[i] for i in pt.argmax(1)]}) \
      .to_csv(f'{out}/{cfg["tag"]}_submission.csv', index=False)
    print('saved:', best_path, last_path, f'{cfg["tag"]}_probs.npz', f'{cfg["tag"]}_submission.csv')
    return model


if __name__ == '__main__':
    main()